# 02 · Volume Profile & Point of Control — Finance Concept

**Contexto:** Peter Steidlmayer (CME, 1980s) observó que los gráficos de precio tradicionales no mostraban *cuánto* se negociaba a cada precio — solo el movimiento. El **Market Profile** nació para responder: ¿a qué precio se concentra la mayor actividad del mercado?

**Campo de origen:** CME Group · traders de futuros · market makers de opciones  
**Dataset:** E-mini S&P 500 futures (ES) — precios y volumen intradiario (simulado con estadísticos reales)

> Para datos reales: Interactive Brokers API, Polygon.io, o `yfinance` (datos diarios OHLCV)

---

## Marco teórico

### Point of Control (POC)

El **POC** es el precio con mayor volumen acumulado en la sesión — actúa como nivel de equilibrio entre compradores y vendedores:

$$POC = \arg\max_{b} \; V_{bin}(b) \qquad V_{bin}(b) = \sum_{i:\, P_i \in bin_b} V_i$$

### Value Area (VA)

Zona de precios que concentra el **70% del volumen total** — inspirada en $\mu \pm \sigma$ de la Normal (68.3%, redondeado a 70% por Steidlmayer):

$$VA = \left\{\, b \;:\; \sum_{b \in VA} V_{bin}(b) \geq 0.70 \times V_{total} \right\}$$

Con límites **VAH** (Value Area High) y **VAL** (Value Area Low).

### Regla de apertura — estrategia de trading

- Apertura **dentro** del VA del día anterior → mercado en equilibrio → probable rango lateral  
- Apertura **fuera** del VA y **no regresa** → ruptura → tendencia intradiaria  
- Apertura **fuera** del VA y **regresa** al VA en los primeros 30 min → reversión hacia el POC  

### Anchura del Perfil (Profile Width) como proxy de volatilidad

$$PW = \frac{VAH - VAL}{POC}$$

PW alto → sesión volátil, rango amplio. PW bajo → consolidación, poco movimiento.

**Referencias:** Steidlmayer, J.P. & Koy, S. (1986). *Markets & Market Logic*. Chicago: Porcupine Press. Dalton, J.F. et al. (1990). *Mind Over Markets*. Probus Publishing.

In [ ]:
# ── IMPORTS ───────────────────────────────────────────────────────────────────
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy import stats
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)

C = dict(
    price='#1E293B',  poc='#DC2626',    va='#DBEAFE',
    vah='#1D4ED8',    val='#1D4ED8',    vol='#334155',
    buy='#15803D',    sell='#B91C1C',   neutral='#64748B',
    profile='#93C5FD'
)
np.random.seed(42)
print('✓ OK')

In [ ]:
# ── DATOS ─────────────────────────────────────────────────────────────────────
# Dataset: E-mini S&P 500 futures (ES) — datos OHLCV diarios
# Fuente real: https://finance.yahoo.com/quote/ES=F  →  yf.download('ES=F')
#              Interactive Brokers TWS API para datos intradiarios
#
# Simulación: 252 días hábiles con estadísticos reales del ES 2023
#   Precio inicial: 4200  ·  retorno diario μ≈+0.04%  ·  σ≈0.85%
#   Volumen diario medio: ~1.2M contratos  ·  CV volumen ≈ 0.35

n_days = 252
dates  = pd.bdate_range('2023-01-02', periods=n_days)

# Precio de cierre con drift y volatilidad clusterizada
returns = []
vol = 0.0085
for i in range(n_days):
    vol = np.sqrt(0.05 + 0.15*(returns[-1]**2 if returns else 0) + 0.80*vol**2)
    vol = np.clip(vol, 0.004, 0.025)
    returns.append(np.random.normal(0.0004, vol))

close = 4200 * np.cumprod(1 + np.array(returns))

# OHLC sintético: open±0.3%, high=close+range/2, low=close-range/2
daily_range = close * np.abs(np.random.normal(0, 0.008, n_days))
open_  = close * (1 + np.random.normal(0, 0.003, n_days))
high   = np.maximum(close, open_) + daily_range * 0.4
low    = np.minimum(close, open_) - daily_range * 0.4

# Volumen: media 1.2M, correlacionado negativamente con precio (más vol en caídas)
volume = np.maximum(
    1_200_000 * (1 - 0.3*np.array(returns)/np.std(returns)) *
    np.random.lognormal(0, 0.30, n_days), 300_000
).astype(int)

df = pd.DataFrame({'open': open_, 'high': high, 'low': low,
                   'close': close, 'volume': volume}, index=dates)

print(f'Serie  : {len(df)} días hábiles ({df.index[0].date()} → {df.index[-1].date()})')
print(f'Precio : ${df.close.min():.0f} – ${df.close.max():.0f}')
print(f'Volumen: {df.volume.mean()/1e6:.2f}M contratos/día (media)')

## Mini-EDA

In [ ]:
# ── EDA 1/2 — Estadísticos clave ─────────────────────────────────────────────
ret = df.close.pct_change().dropna()

stats_tbl = {
    'n días'            : (len(df),              '1 año hábil completo'),
    'Precio inicial'    : (f'${df.close.iloc[0]:.0f}',  'nivel de entrada'),
    'Precio final'      : (f'${df.close.iloc[-1]:.0f}', 'nivel de salida'),
    'Retorno total'     : (f'{(df.close.iloc[-1]/df.close.iloc[0]-1):.1%}', ''),
    'Vol anualizada'    : (f'{ret.std()*np.sqrt(252):.1%}', 'σ de retornos diarios × √252'),
    'Volumen medio/día' : (f'{df.volume.mean()/1e6:.2f}M',  'contratos E-mini'),
    'CV volumen'        : (f'{df.volume.std()/df.volume.mean():.3f}', '< 0.5 → variabilidad moderada'),
    'Skew retornos'     : (f'{ret.skew():.3f}',  'negativo → cola izquierda (crashes)'),
    'Kurt retornos'     : (f'{ret.kurt():.3f}',  '> 0 → fat tails → Normal subestima riesgo'),
    'Rango diario medio': (f'{((df.high-df.low)/df.close).mean():.2%}', 'high-low / close'),
}

print(f'{"Métrica":<22} {"Valor":<14} {"Nota"}')
print('─' * 65)
for k, (v, note) in stats_tbl.items():
    print(f'{k:<22} {str(v):<14} {note}')

In [ ]:
# ── EDA 2/2 — Precio + volumen ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1], 'hspace': 0.06})

ax1 = axes[0]
ax1.plot(df.index, df.close, color=C['price'], lw=1.0, label='ES precio cierre')
ax1.fill_between(df.index, df.high, df.low, alpha=0.08, color=C['profile'])
ax1.set_ylabel('Precio (USD)')
ax1.set_title('E-mini S&P 500 (ES) — Precio + Volumen diario (simulado 2023)', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(axis='y', alpha=0.3)

ax2 = axes[1]
colors_vol = np.where(df.close >= df.open, C['buy'], C['sell'])
ax2.bar(df.index, df.volume/1e6, color=colors_vol, alpha=0.7, width=0.8)
ax2.set_ylabel('Volumen (M)')
ax2.set_xlabel('Fecha')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_eda.png', dpi=130, bbox_inches='tight')
plt.show()
print('✓ data/finance_eda.png')

In [ ]:
# ── FUNCIONES CORE — Volume Profile ──────────────────────────────────────────

def build_volume_profile(prices, volumes, n_bins=30):
    """
    Construye el Volume Profile de una serie de precios con sus volúmenes.

    Parameters
    ----------
    prices  : array-like de precios (close, o midpoint high-low)
    volumes : array-like de volúmenes correspondientes
    n_bins  : número de niveles de precio (resolución del perfil)

    Returns
    -------
    dict con: bin_centers, bin_volumes, poc, vah, val, value_area_pct
    """
    prices  = np.array(prices)
    volumes = np.array(volumes)

    p_min, p_max = prices.min(), prices.max()
    bin_edges   = np.linspace(p_min, p_max, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    # Acumular volumen en cada bin
    bin_volumes = np.zeros(n_bins)
    for p, v in zip(prices, volumes):
        idx = np.searchsorted(bin_edges[1:], p, side='left')
        idx = min(idx, n_bins - 1)
        bin_volumes[idx] += v

    total_vol = bin_volumes.sum()

    # POC — bin con mayor volumen
    poc_idx = np.argmax(bin_volumes)
    poc     = bin_centers[poc_idx]

    # Value Area — 70% del volumen total, expandiendo desde el POC
    sorted_idx = np.argsort(bin_volumes)[::-1]  # bins ordenados por volumen desc
    va_idx, acum = [], 0.0
    for i in sorted_idx:
        va_idx.append(i)
        acum += bin_volumes[i]
        if acum >= 0.70 * total_vol:
            break

    vah = bin_centers[max(va_idx)]
    val = bin_centers[min(va_idx)]
    value_area_pct = acum / total_vol

    return {
        'bin_centers'    : bin_centers,
        'bin_volumes'    : bin_volumes,
        'poc'            : poc,
        'vah'            : vah,
        'val'            : val,
        'value_area_pct' : value_area_pct,
        'total_volume'   : total_vol,
        'profile_width'  : (vah - val) / poc,
    }


def classify_open(open_price, prev_vah, prev_val, prev_poc):
    """
    Clasifica la apertura del día según la teoría de Market Profile.
    Retorna la señal de trading esperada.
    """
    if prev_val <= open_price <= prev_vah:
        return 'dentro_va', '⬜ Dentro VA → rango lateral esperado'
    elif open_price > prev_vah:
        return 'arriba_va', '🟢 Sobre VAH → posible tendencia alcista'
    else:
        return 'abajo_va', '🔴 Bajo VAL → posible tendencia bajista'


# Verificar con datos de un día
profile_full = build_volume_profile(
    prices  = df.close.values,
    volumes = df.volume.values,
    n_bins  = 40
)

print('Volume Profile — año completo')
print(f'  POC : ${profile_full["poc"]:,.1f}')
print(f'  VAH : ${profile_full["vah"]:,.1f}')
print(f'  VAL : ${profile_full["val"]:,.1f}')
print(f'  VA  : {profile_full["value_area_pct"]:.1%} del volumen total cubierto')
print(f'  PW  : {profile_full["profile_width"]:.3f} (anchura relativa del VA)')

In [ ]:
# ── VOLUME PROFILE ROLLING — trimestral ──────────────────────────────────────
# Calculamos el perfil en ventanas rodantes de 63 días (≈ 1 trimestre)

window = 63
rolling_profiles = []

for i in range(window, len(df) + 1):
    window_df = df.iloc[i - window:i]
    p = build_volume_profile(
        prices  = window_df.close.values,
        volumes = window_df.volume.values,
        n_bins  = 30
    )
    rolling_profiles.append({
        'date'          : df.index[i - 1],
        'poc'           : p['poc'],
        'vah'           : p['vah'],
        'val'           : p['val'],
        'profile_width' : p['profile_width'],
    })

rp = pd.DataFrame(rolling_profiles).set_index('date')

print(f'Perfiles rodantes calculados: {len(rp)}')
print(rp.tail(5).round(1).to_string())

In [ ]:
# ── DASHBOARD PRINCIPAL ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 13))
fig.suptitle(
    'Volume Profile & Point of Control — E-mini S&P 500 (ES)\n'
    'Mercado de origen: CME Group · Futuros sobre índices (simulado 2023)',
    fontsize=13, fontweight='bold', y=0.99
)

gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.3,
                        width_ratios=[3, 1], height_ratios=[2, 1])

# ─ Panel 1: Precio + VA rodante ─────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(df.index, df.close, color=C['price'], lw=1.0, label='ES precio cierre', zorder=3)
ax1.fill_between(rp.index, rp.vah, rp.val,
                 alpha=0.15, color=C['profile'], label='Value Area rodante (70% vol, 63d)')
ax1.plot(rp.index, rp.poc, color=C['poc'], lw=1.2, ls='--', label='POC rodante')
ax1.plot(rp.index, rp.vah, color=C['vah'], lw=0.7, alpha=0.8)
ax1.plot(rp.index, rp.val, color=C['val'], lw=0.7, alpha=0.8)
ax1.set_ylabel('Precio USD')
ax1.set_title('Precio + Value Area rodante (ventana 63 días)', loc='left', fontsize=10)
ax1.legend(fontsize=8, loc='upper left')
ax1.grid(axis='y', alpha=0.3)

# ─ Panel 2: Volume Profile vertical (año completo) ───────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
bc = profile_full['bin_centers']
bv = profile_full['bin_volumes'] / 1e6  # en millones
poc = profile_full['poc']
vah = profile_full['vah']
val = profile_full['val']

# Colorear: rojo=POC, azul=VA, gris=fuera del VA
bar_colors = []
for center in bc:
    if abs(center - poc) < (bc[1] - bc[0]) / 2:
        bar_colors.append(C['poc'])
    elif val <= center <= vah:
        bar_colors.append(C['profile'])
    else:
        bar_colors.append(C['neutral'])

ax2.barh(bc, bv, height=(bc[1]-bc[0])*0.85,
         color=bar_colors, edgecolor='white', linewidth=0.3)
ax2.axhline(poc, color=C['poc'],  lw=1.5, ls='--', label=f'POC ${poc:.0f}')
ax2.axhline(vah, color=C['vah'],  lw=1.0, ls=':',  label=f'VAH ${vah:.0f}')
ax2.axhline(val, color=C['val'],  lw=1.0, ls=':',  label=f'VAL ${val:.0f}')
ax2.set_xlabel('Volumen (M contratos)')
ax2.set_title('Volume Profile\naño completo', loc='left', fontsize=10)
ax2.legend(fontsize=7)
ax2.grid(axis='x', alpha=0.3)

# ─ Panel 3: POC vs Precio — señal de soporte/resistencia ────────────────────
ax3 = fig.add_subplot(gs[1, 0])
spread = df.close.loc[rp.index] - rp.poc
ax3.fill_between(rp.index, spread, 0,
                 where=spread >= 0, alpha=0.4, color=C['buy'],  label='Precio > POC')
ax3.fill_between(rp.index, spread, 0,
                 where=spread <  0, alpha=0.4, color=C['sell'], label='Precio < POC')
ax3.axhline(0, color=C['poc'], lw=0.8, ls='--')
ax3.set_ylabel('Precio − POC (USD)')
ax3.set_xlabel('Fecha')
ax3.set_title('Spread Precio − POC: cruces = cambio de régimen', loc='left', fontsize=10)
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# ─ Panel 4: Profile Width — volatilidad implícita del perfil ────────────────
ax4 = fig.add_subplot(gs[1, 1])
ax4.fill_between(rp.index, rp.profile_width * 100, alpha=0.5, color=C['vah'])
ax4.plot(rp.index, rp.profile_width * 100, color=C['vah'], lw=0.9)
ax4.axhline(rp.profile_width.quantile(0.80) * 100,
            color=C['sell'], lw=0.8, ls='--', label='p80 (alta volatilidad)')
ax4.axhline(rp.profile_width.quantile(0.20) * 100,
            color=C['buy'],  lw=0.8, ls='--', label='p20 (baja volatilidad)')
ax4.set_ylabel('Profile Width (%)')
ax4.set_xlabel('Fecha')
ax4.set_title('Profile Width\n(volatilidad del perfil)', loc='left', fontsize=10)
ax4.legend(fontsize=8)
ax4.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('data/finance_dashboard.png', dpi=140, bbox_inches='tight')
plt.show()
print('✓ data/finance_dashboard.png')

In [ ]:
# ── ANÁLISIS — Regla de apertura ─────────────────────────────────────────────
# Clasificar cada apertura según el VA del día anterior

signals = []
for i in range(1, len(df)):
    today     = df.iloc[i]
    prev_prof = build_volume_profile(
        prices  = df.iloc[max(0, i-21):i].close.values,
        volumes = df.iloc[max(0, i-21):i].volume.values,
        n_bins  = 20
    )
    signal_type, label = classify_open(
        today.open, prev_prof['vah'], prev_prof['val'], prev_prof['poc']
    )
    # Retorno del día
    day_return = (today.close - today.open) / today.open
    signals.append({
        'date'        : df.index[i],
        'signal_type' : signal_type,
        'label'       : label,
        'day_return'  : day_return,
        'poc'         : prev_prof['poc'],
    })

sig_df = pd.DataFrame(signals).set_index('date')

print('── Distribución de tipos de apertura ─────────────────')
print(sig_df.signal_type.value_counts().to_string())

print('\n── Retorno medio del día por tipo de apertura ────────')
print(sig_df.groupby('signal_type')['day_return'].agg(['mean','std','count']).round(4).to_string())

print('\n── Interpretación ────────────────────────────────────')
for stype, group in sig_df.groupby('signal_type'):
    mean_ret = group.day_return.mean() * 100
    print(f'{stype:15s}: retorno medio del día {mean_ret:+.2f}%')

In [ ]:
# ── EXPORTAR ─────────────────────────────────────────────────────────────────
sig_df.to_csv('data/finance_signals.csv')
rp.to_csv('data/finance_rolling_profiles.csv')
print('✓ data/finance_signals.csv')
print('✓ data/finance_rolling_profiles.csv')
print('✓ data/finance_eda.png')
print('✓ data/finance_dashboard.png')

## Conclusiones — contexto financiero

| Concepto | En futuros ES | Uso práctico |
|----------|--------------|-------------|
| **POC** | Precio con mayor volumen | Nivel de soporte/resistencia más importante del día |
| **VAH** | Límite superior del 70% | Resistencia: precio tiende a rebotar aquí |
| **VAL** | Límite inferior del 70% | Soporte: precio tiende a rebotar aquí |
| **Apertura fuera VA** | Evento inusual | Alta probabilidad de tendencia intradiaria |
| **Profile Width** | Anchura del VA | Proxy de volatilidad no paramétrico |
| **Spread P−POC** | Precio vs. equilibrio | Cruces del cero = cambio de régimen de mercado |

**Próximo paso:** `2_Supply_Adaptation.ipynb` — la misma lógica aplicada a demanda de SKUs de Alicorp, donde precio → demanda y volumen → frecuencia de semanas.